**Malaria Dataset — 最佳正確率模型**

Dataset：Malaria Cell Images（TFDS）

Task：Binary Classification（Parasitized / Uninfected）

Model：自創 Deep CNN + Batch Normalization + Dropout

Loss：Binary Cross Entropy

1. **套件匯入**

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt


2. 載入 Malaria Dataset

In [ ]:
# 載入 Malaria Dataset
(train_ds, val_ds, test_ds), info = tfds.load(
    'malaria',
    split=['train[:70%]', 'train[70%:85%]', 'train[85%:]'],
    as_supervised=True,
    with_info=True
)

class_names = info.features['label'].names
num_classes = 1   # Binary classification只需要一個輸出神經元


3. 資料前處理（顯微鏡影像）

In [ ]:
def preprocess(image, label):
    # 將像素值轉為 float 並正規化到 0~1
    # 有助於模型訓練穩定、加快收斂
    image = tf.cast(image, tf.float32) / 255.0
    # 統一影像尺寸，讓模型輸入固定大小
    image = tf.image.resize(image, (64, 64))
    # 將 label 轉為 float，配合 binary cross entropy
    label = tf.cast(label, tf.float32)
    return image, label

batch_size = 32

train_ds = train_ds.map(preprocess).shuffle(1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.map(preprocess).batch(batch_size).prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds.map(preprocess).batch(batch_size).prefetch(tf.data.AUTOTUNE)


4. 自創最佳模型（Best Custom CNN for Malaria）

In [ ]:
def build_best_model():
    model = tf.keras.Sequential([

        # Block 1
        tf.keras.layers.Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(64,64,3)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Dropout(0.25),

        # Block 2
        tf.keras.layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Dropout(0.25),

        # Block 3
        tf.keras.layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2,2),

        # Fully Connected
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    return model


5. 編譯與訓練模型

In [ ]:
model = build_best_model()

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds
)


Epoch 1/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 539s 883ms/step - accuracy: 0.7392 - loss: 0.5732 - val_accuracy: 0.8718 - val_loss: 0.4956
Epoch 2/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 565s 890ms/step - accuracy: 0.9451 - loss: 0.1610 - val_accuracy: 0.7147 - val_loss: 2.2864
Epoch 3/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 534s 885ms/step - accuracy: 0.9512 - loss: 0.1500 - val_accuracy: 0.9560 - val_loss: 0.1345
Epoch 4/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 540s 894ms/step - accuracy: 0.9523 - loss: 0.1383 - val_accuracy: 0.9475 - val_loss: 0.1689
Epoch 5/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 535s 887ms/step - accuracy: 0.9550 - loss: 0.1358 - val_accuracy: 0.9412 - val_loss: 0.2856
Epoch 6/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 537s 890ms/step - accuracy: 0.9542 - loss: 0.1308 - val_accuracy: 0.9279 - val_loss: 0.2357
Epoch 7/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 571s 904ms/step - accuracy: 0.9577 - loss: 0.1253 - val_accuracy: 0.9458 - val_loss: 0.1934
Epoch 8/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 550s 885ms/step - accuracy: 0.9579 -

6. **最終測試集評估**

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Malaria 最佳模型測試集準確率：{test_accuracy * 100:.2f}%")


130/130 ━━━━━━━━━━━━━━━━━━━━ 43s 325ms/step - accuracy: 0.9182 - loss: 0.3172
Malaria 最佳模型測試集準確率：92.31%
